In [95]:
# -*- coding: utf-8 -*-
"""
Incendis forestals a Catalunya (1998-2024) — Visualitzacions Finals Q1-Q4
Information Visualization | Projecte 1
"""

import copy
import json
import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

DATA_PATH = "data/incendis_final.csv"
COMARQUES_MAP_PATH = 'data/divisions-administratives-v2r2-comarques-500000-20260120.json'

df = pd.read_csv(DATA_PATH)
top_causes = df['Causa'].value_counts().nlargest(7).index
df['CausaAgrupada'] = df['Causa'].where(df['Causa'].isin(top_causes), 'Altres')
causa_domain = sorted(df['CausaAgrupada'].unique())
causa_scale = alt.Scale(domain=causa_domain, scheme='dark2')


# --- GeoJSON i càlcul d'aspect ratio real ---
geojson_path = (
    COMARQUES_MAP_PATH
)
with open(geojson_path, encoding='utf-8') as f:
    geo_dict = json.load(f)
comarques_geo = alt.Data(values=geo_dict['features'])
noms_geojson = {feat['properties']['NOMCOMAR'] for feat in geo_dict['features']}


def _calcular_bbox(geo_d):
    xs, ys = [], []

    def recorre(coords):
        if isinstance(coords[0], (int, float)):
            xs.append(coords[0])
            ys.append(coords[1])
        else:
            for c in coords:
                recorre(c)

    for feat in geo_d['features']:
        recorre(feat['geometry']['coordinates'])
    return min(xs), max(xs), min(ys), max(ys)


_minx, _maxx, _miny, _maxy = _calcular_bbox(geo_dict)
ASPECT_RATIO = (_maxx - _minx) / (_maxy - _miny)


def mida_mapa(amplada):
    return amplada, round(amplada / ASPECT_RATIO)


PROJECCIO = 'identity'
REFLECTEIX_Y = True


def titol_rt(titol, subtitol):
    return alt.TitleParams(
        text=titol,
        subtitle=subtitol,
        anchor='start',
        fontSize=20,
        fontWeight='bold',
        subtitleFontSize=13,
        subtitleColor='#555555',
        offset=12
    )

In [117]:
"""Q2 - Evolution of the causes of fires: five Altair proposals.

Input: the cleaned fires table (OpenRefine export) with at least the columns
    Campania, Origen, Causa, SuperficieTotalForestal
Usage in Colab / Streamlit:
    from q2_charts import prepare, chart1, chart2, chart3, chart4, chart5
    d = prepare(df)            # df = your cleaned DataFrame
    chart1(d)                  # etc.
"""
import altair as alt
import pandas as pd


NAMED_CAUSA_COLORS = {
    "Intentional": "tomato",
    "Other unintentional causes": "darkorange",
    "Agricultural or livestock burning": "mediumseagreen",
    "Power lines": "gold",
    "Lightning": "royalblue",
    "Smokers": "goldenrod",
    "Unknown": "mediumpurple",
    "Other": "darkgray",
}

CAUSA_EN = {
    "Intencionado": "Intentional",
    "Altres": "Other",
    "Desconocida": "Unknown",
    "Quemas agrícolas y ganaderas": "Agricultural or livestock burning",
    "Rayo": "Lightning",
    "Otras causas no intencionales": "Other unintentional causes",
    "Causas por líneas eléctricas": "Power lines",
    "Fumadores": "Smokers",
}
CAUSA_ORDER = [
    "Intentional",
    "Other unintentional causes",
    "Agricultural or livestock burning",
    "Power lines",
    "Lightning",
    "Smokers",
    "Unknown",
    "Other",
]

LEGEND = alt.Legend(
    title="Cause of fire",
    orient="top",
    direction="horizontal",
    labelExpr="{" + ", ".join(f"'{k}': '{v}'" for k, v in CAUSA_EN.items()) + "}[datum.label] || datum.label",
)


def beeswarm_data(d, per_dot=5, cols=7, spread=0.9, layout="stacked"):
    """One row per dot (1 dot = `per_dot` fires; cumulative rounding per year)."""
    dx = spread / cols
    mapped = d["CausaAgrupada"].dropna().astype(str).map(lambda c: CAUSA_EN.get(c, c))

    freq_order = mapped.value_counts().index.tolist()
    ordered_causes = [c for c in CAUSA_ORDER if c in freq_order] + [
        c for c in freq_order if c not in CAUSA_ORDER
    ]

    temp = d.loc[mapped.index, ["Any"]].copy()
    temp["cause_en"] = mapped

    counts_all = (
        temp.groupby(["Any", "cause_en"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=ordered_causes, fill_value=0)
    )

    dots = {}
    for yr, counts in counts_all.iterrows():
        cum, prev = counts.cumsum(), 0
        for causa in ordered_causes:
            end = int(round(cum[causa] / per_dot))
            dots[(yr, causa)] = (end - prev, int(counts[causa]), prev)
            prev = end

    year_totals = {}
    for (yr, _causa), (_nd, fires, _first) in dots.items():
        year_totals[yr] = year_totals.get(yr, 0) + fires

    lane_rows = max((-(-nd // cols) for nd, _, _ in dots.values()), default=1)
    lane_h = lane_rows + 2

    out = []
    for (yr, causa), (nd, fires, first) in dots.items():
        for j in range(nd):
            k = first + j
            r, c = divmod(k, cols)
            x = yr + (c - (cols - 1) / 2) * dx + (dx / 4 if r % 2 == 0 else -dx / 4)
            y = r * 0.866
            total_yr = year_totals.get(yr, 0)
            percent_yr = (fires / total_yr) if total_yr > 0 else 0.0

            out.append({
                "Any": yr,
                "cause": causa,
                "x": x,
                "y": y,
                "fires": int(fires),
                "year_percent": percent_yr,
            })

    out_df = pd.DataFrame(out)
    out_df.attrs["lane_h"] = lane_h
    out_df.attrs["causes"] = ordered_causes
    return out_df


def chart6(d, per_dot=5, cols=7, width=1500, layout="stacked"):
    """Wide beeswarm of fires per year, coloured by cause. 1 dot = `per_dot` fires."""
    b = beeswarm_data(d, per_dot, cols, layout=layout)

    print("Causes in input d:", d["CausaAgrupada"].value_counts(dropna=False))
    print("Causes in beeswarm b:", b["cause"].value_counts(dropna=False))

    causes = b.attrs["causes"]
    cause_scale = alt.Scale(
        domain=causes,
        range=[NAMED_CAUSA_COLORS.get(c, "slategray") for c in causes],
    )
    legend = alt.Legend(
        title="Cause of fire",
        orient="top",
        direction="horizontal",
    )
    yr_min = int(d["Any"].min())
    yr_max = int(d["Any"].max())
    n_years = max(yr_max - yr_min + 1, 1)

    px_year = width / n_years
    dx_px = px_year * 0.9 / cols
    ymax = float(b["y"].max()) + 1 if not b.empty else 1.0
    height = max(int(ymax * dx_px) + 30, 120)
    size = (dx_px * 0.92) ** 2

    xenc = alt.X(
        "x:Q",
        title="Year",
        scale=alt.Scale(domain=[yr_min - 0.5, yr_max + 0.5], nice=False, zero=False),
        axis=alt.Axis(format="d", values=list(range(yr_min, yr_max + 1)), grid=False, labelFontSize=11),
    )
    title = alt.TitleParams(text=f"Fires per year by cause (1 dot = {per_dot} fires)", anchor="middle")

    yenc = alt.Y(
        "y:Q",
        title="Fires per year (approx.)",
        scale=alt.Scale(domain=[-1, ymax], nice=False),
        axis=alt.Axis(
            grid=True,
            gridDash=[2, 3],
            labelExpr=f"round(datum.value / 0.866 * {cols * per_dot})",
            values=[0.866 * v / (cols * per_dot) for v in range(0, int(ymax / 0.866 * cols * per_dot) + 1, 100 if per_dot <= 5 else 200)],
        ),
    )

    dots = (
        alt.Chart(b)
        .mark_circle(size=size, opacity=1, stroke="white", strokeWidth=0.4)
        .encode(
            x=xenc,
            y=yenc,
            color=alt.Color("cause:N", scale=cause_scale, legend=LEGEND),
            tooltip=[
                alt.Tooltip("Any:O", title="Year"),
                alt.Tooltip("cause:N", title="Cause of fire"),
                alt.Tooltip("fires:Q", title="Fires of this cause"),
                alt.Tooltip("year_percent:Q", title="% of year's total", format=".1%"),
            ],
        )
    )
    return dots.properties(width=width, height=height, title=title)


q2 = chart6(df, per_dot=5, cols=6, width=1900)
q2.save("outputs/Q2.json")
q2


Causes in input d: CausaAgrupada
Intencionado                     3765
Desconocida                      2984
Altres                           2896
Quemas agrícolas y ganaderas     2047
Rayo                             1573
Otras causas no intencionales    1407
Causas por líneas eléctricas      889
Fumadores                         857
Name: count, dtype: int64
Causes in beeswarm b: cause
Intentional                          752
Unknown                              598
Other                                578
Agricultural or livestock burning    409
Lightning                            318
Other unintentional causes           282
Power lines                          176
Smokers                              170
Name: count, dtype: int64


alt.Chart(...)

In [56]:
counts_raw = df["CausaAgrupada"].value_counts(dropna=False)
counts_plotted = df["CausaAgrupada"].dropna().astype(str).value_counts()

print("ORDRE_CAUSA:", ORDRE_CAUSA)
print("Top counts in CausaAgrupada:\n", counts_raw.head(10))
print("Exact match with ORDRE_CAUSA?:", ORDRE_CAUSA == counts_plotted.index.tolist())

ORDRE_CAUSA: ['Intencionado', 'Altres', 'Desconocida', 'Quemas agrícolas y ganaderas', 'Rayo', 'Otras causas no intencionales', 'Causas por líneas eléctricas']
Top counts in CausaAgrupada:
 CausaAgrupada
Intencionado                     3765
Altres                           3753
Desconocida                      2984
Quemas agrícolas y ganaderas     2047
Rayo                             1573
Otras causas no intencionales    1407
Causas por líneas eléctricas      889
Name: count, dtype: int64
Exact match with ORDRE_CAUSA?: True
